In [31]:
import tkinter as tk
import numpy as np

root = tk.Tk()
root.title("Gaze Focus Zones")

canvas_w, canvas_h = 800, 600
canvas = tk.Canvas(root, width=canvas_w, height=canvas_h, bg="white")
canvas.pack()

zeta_entry = tk.Entry(root)
zeta_entry.insert(0, "180")  # أقصى زاوية افتراضيًا
zeta_entry.pack()

# نقطة بداية النظر ومتجه النظر
gaze_start = np.array([canvas_w // 2, canvas_h - 50])
gaze_direction = np.array([0, -1])
gaze_end = gaze_start + gaze_direction * 550

targets = []

def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot = np.clip(np.dot(v1_u, v2_u), -1.0, 1.0)
    return np.degrees(np.arccos(dot))

def draw_gaze():
    canvas.create_oval(gaze_start[0]-5, gaze_start[1]-5,
                       gaze_start[0]+5, gaze_start[1]+5, fill="blue")
    canvas.create_line(*gaze_start, *gaze_end, fill="blue", width=2, arrow=tk.LAST)

def add_target(event):
    targets.append((event.x, event.y))
    canvas.create_oval(event.x-5, event.y-5, event.x+5, event.y+5, outline="gray")

canvas.bind("<Button-1>", add_target)

def analyze():
    canvas.delete("all")
    draw_gaze()

    try:
        zeta = float(zeta_entry.get())
    except:
        print("❌ أدخل زاوية صحيحة")
        return

    for tx, ty in targets:
        target_point = np.array([tx, ty])
        vec_to_target = target_point - gaze_start
        angle = angle_between(gaze_end - gaze_start, vec_to_target)

        if angle <= 20:
            triangle_color = "#cc0000"  # أحمر
            label = "Focus"
        elif angle <= 30:
            triangle_color = "#ff6600"  # برتقالي
            label = "Word"
        elif angle <= 60:
            triangle_color = "#ffcc00"  # أصفر
            label = "Color"
        elif angle <= 124:
            triangle_color = "#66cc66"  # أخضر
            label = "Rotation"
        elif angle <= 180:
            triangle_color = "#99f3ff"  # أزرق
            label = "Peripheral"
        else:
            triangle_color = None
            label = "None"

        # نظلل المساحة إذا كان التركيز عالي (زاوية ≤ 20)
        if angle <= 20:
            canvas.create_polygon(
                gaze_start[0], gaze_start[1],
                gaze_end[0], gaze_end[1],
                tx, ty,
                fill=triangle_color,
                outline=""
            )

        # نرسم الخط من العين للتارجت
        canvas.create_line(*gaze_start, tx, ty, fill=triangle_color, dash=(3, 2))
        # نرسم التارجت
        canvas.create_oval(tx-6, ty-6, tx+6, ty+6, fill=triangle_color)
        # نكتب الزاوية
        canvas.create_text(tx, ty - 15, text=f"{int(angle)}° ({label})",
                        font=("Arial", 10), fill="black")


def refresh():
    canvas.delete("all")
    targets.clear()
    draw_gaze()

tk.Button(root, text="Analyze", command=analyze).pack()
tk.Button(root, text="Refresh", command=refresh).pack()

# نرسم gaze line مبدئياً
draw_gaze()

root.mainloop()


In [1]:
import numpy as np
import cv2
import mediapipe as mp
import time

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.5,min_tracking_confidence=0.5)

mp_drawing = mp.solutions.drawing_utils

drawing_spec = mp_drawing.DrawingSpec(color=(128,0,128),thickness=2,circle_radius=1)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, image = cap.read()

    start = time.time()

    image = cv2.cvtColor(cv2.flip(image,1),cv2.COLOR_BGR2RGB) #flipped for selfie view

    image.flags.writeable = False

    results = face_mesh.process(image)

    image.flags.writeable = True

    image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)

    img_h , img_w, img_c = image.shape
    face_2d = []
    face_3d = []

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            for idx, lm in enumerate(face_landmarks.landmark):
                if idx == 33 or idx == 263 or idx ==1 or idx == 61 or idx == 291 or idx==199:
                    if idx ==1:
                        nose_2d = (lm.x * img_w,lm.y * img_h)
                        nose_3d = (lm.x * img_w,lm.y * img_h,lm.z * 3000)
                    x,y = int(lm.x * img_w),int(lm.y * img_h)

                    face_2d.append([x,y])
                    face_3d.append(([x,y,lm.z]))


            #Get 2d Coord
            face_2d = np.array(face_2d,dtype=np.float64)

            face_3d = np.array(face_3d,dtype=np.float64)

            focal_length = 1 * img_w

            cam_matrix = np.array([[focal_length,0,img_h/2],
                                  [0,focal_length,img_w/2],
                                  [0,0,1]])
            distortion_matrix = np.zeros((4,1),dtype=np.float64)

            success,rotation_vec,translation_vec = cv2.solvePnP(face_3d,face_2d,cam_matrix,distortion_matrix)


            #getting rotational of face
            rmat,jac = cv2.Rodrigues(rotation_vec)

            angles,mtxR,mtxQ,Qx,Qy,Qz = cv2.RQDecomp3x3(rmat)

            x = angles[0] * 360
            y = angles[1] * 360
            z = angles[2] * 360

            #here based on axis rot angle is calculated
            if y < -10:
                text="Looking Left"
            elif y > 10:
                text="Looking Right"
            elif x < -10:
                text="Looking Down"
            elif x > 10:
                text="Looking Up"
            else:
                text="Forward"

            nose_3d_projection,jacobian = cv2.projectPoints(nose_3d,rotation_vec,translation_vec,cam_matrix,distortion_matrix)

            p1 = (int(nose_2d[0]),int(nose_2d[1]))
            p2 = (int(nose_2d[0] + y*10), int(nose_2d[1] -x *10))

            cv2.line(image,p1,p2,(255,0,0),3)

            cv2.putText(image,text,(20,50),cv2.FONT_HERSHEY_SIMPLEX,2,(0,255,0),2)
            cv2.putText(image,"x: " + str(np.round(x,2)),(500,50),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)
            cv2.putText(image,"y: "+ str(np.round(y,2)),(500,100),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)
            cv2.putText(image,"z: "+ str(np.round(z, 2)), (500, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)


        end = time.time()
        totalTime = end-start

        

        
    cv2.imshow('Head Pose Detection',image)
    if cv2.waitKey(5) & 0xFF ==27:
        break
cap.release()

: 